# ML Assignment 2 - Online Shoppers Purchase-Intent Classifiers

End-to-end run of the assignment on the UCI **Online Shoppers Purchasing Intention** dataset
(12,330 sessions, 17 features, target `Revenue`):

1. clean the data and make a stratified 80/20 split,
2. tune and train Logistic Regression, Decision Tree, KNN, Naive Bayes and SVM,
3. evaluate every model with Accuracy, AUC, Precision, Recall, F1 and MCC on the hold-out split,
4. save the fitted pipelines (`.joblib`) and the hold-out file (`test_data.csv`).

All logic lives in `train_models.py` and `shopper_utils.py`, which the Streamlit app (`app.py`) also uses.

**How to run:** restart the kernel and run all cells (JupyterLab: *Kernel > Restart Kernel and Run All Cells*).
The notebook is self-contained:

* The **environment check** installs the pinned library versions from `requirements.txt` only if they are missing or different.
  If it has to change a library that this kernel has already loaded, it restarts the kernel once - just run all cells again.
* The **project files** (`shopper_utils.py`, `train_models.py`, `data/online_shoppers_intention.csv`) are used from the current
  folder or its parents when they are there; otherwise they are unpacked from the compressed copy stored in the collapsed
  cell below, so the notebook also works when it is the only file on the machine.

In [ ]:
# Environment check - standard library only: nothing from numpy / pandas / scikit-learn is imported here.
import json
import os
import subprocess
import sys
import time

PINNED = {"numpy": "2.5.4", "scipy": "1.18.1", "pandas": "3.1.0", "matplotlib": "3.11.2", "joblib": "1.6.0", "scikit-learn": "1.9.1"}
IMPORT_NAME = {"scikit-learn": "sklearn"}


def versions_on_disk():
    """Installed versions as seen by a fresh interpreter (what the next import would load)."""
    probe = (
        "import importlib.metadata as m, json\n"
        "out = {}\n"
        "for name in " + repr(list(PINNED)) + ":\n"
        "    try:\n"
        "        out[name] = m.version(name)\n"
        "    except m.PackageNotFoundError:\n"
        "        out[name] = None\n"
        "print(json.dumps(out))\n"
    )
    done = subprocess.run([sys.executable, "-c", probe], capture_output=True, text=True, errors="replace")
    return json.loads(done.stdout.strip().splitlines()[-1])


def pip_install(specs, user=False):
    command = [sys.executable, "-m", "pip", "install", "--disable-pip-version-check", "--no-input", "-q"]
    if user:
        command.append("--user")
    try:
        done = subprocess.run(command + list(specs), capture_output=True, text=True, errors="replace", timeout=900)
    except subprocess.TimeoutExpired:
        return False, "pip timed out after 15 minutes"
    return done.returncode == 0, done.stdout + done.stderr


def install(specs):
    ok, log = pip_install(specs)
    if not ok and "ermission" in log:
        ok, log = pip_install(specs, user=True)
    return ok, log


have = versions_on_disk()
if any(have[name] != version for name, version in PINNED.items()):
    print("Installing the pinned library versions (about 1-2 minutes)...")
    ok, log = install([f"{name}=={version}" for name, version in PINNED.items()])
    if not ok:
        print("The pinned versions are not available here; installing the newest compatible versions instead.")
        ok, log = install(list(PINNED))
    if not ok:
        print(log[-1500:])
        raise RuntimeError("pip could not install the required libraries - see the message above.")
    have = versions_on_disk()

missing = [name for name, version in have.items() if version is None]
if missing:
    raise RuntimeError("Still missing after installation: " + ", ".join(missing))

stale = []
for name, version in have.items():
    module = sys.modules.get(IMPORT_NAME.get(name, name))
    loaded = getattr(module, "__version__", None)
    if loaded is not None and loaded != version:
        stale.append(f"{name} (loaded {loaded}, installed {version})")
if stale:
    print("Library versions changed while this kernel was running: " + ", ".join(stale))
    print("The kernel restarts by itself now. When it is back, run all cells again.")
    print("(If it does not restart on its own: Kernel > Restart Kernel, then run all cells.)")
    sys.stdout.flush()
    sys.stderr.flush()
    time.sleep(2)
    os._exit(0)

print("Environment OK | " + " | ".join(f"{name} {version}" for name, version in have.items()))


## 0. Project files

The next cell is collapsed on purpose: it only holds a compressed copy of `shopper_utils.py`, `train_models.py` and
`data/online_shoppers_intention.csv` (the same files as in the GitHub repository). Run it, then the cell after it.

In [ ]:
_BUNDLE = """
...
"""


In [ ]:
# Locate the project files, or unpack the copy stored in the cell above (standard library only).
import base64
import io
import sys
import tempfile
import zipfile
from pathlib import Path

NEEDED = ("shopper_utils.py", "train_models.py", Path("data") / "online_shoppers_intention.csv")


def has_all(folder):
    return all((folder / item).is_file() for item in NEEDED)


here = Path.cwd().resolve()
PROJECT = next((d for d in (here, *list(here.parents)[:3]) if has_all(d)), None)

if PROJECT is None:
    if "_BUNDLE" not in globals():
        raise RuntimeError("Run the collapsed cell above (the one that starts with _BUNDLE) first.")
    blob = base64.b64decode("".join(_BUNDLE.split()))
    for base in (here, Path.home(), Path(tempfile.gettempdir())):
        target = base / "ml_assignment_2_project"
        try:
            with zipfile.ZipFile(io.BytesIO(blob)) as archive:
                archive.extractall(target)
        except OSError:
            continue
        if has_all(target):
            PROJECT = target
            break
if PROJECT is None:
    raise RuntimeError("Could not unpack the project files: no writable folder found.")

for name in ("shopper_utils", "train_models"):
    sys.modules.pop(name, None)
sys.path.insert(0, str(PROJECT))
print("Project folder:", PROJECT)
print("Files:", ", ".join(str(item) for item in NEEDED))


In [ ]:
%matplotlib inline
import platform
import time
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
from sklearn.metrics import ConfusionMatrixDisplay

import shopper_utils as su
import train_models as tm

print("Python", platform.python_version(), "| scikit-learn", sklearn.__version__, "| pandas", pd.__version__)

## 1. Train and evaluate

`OUT_DIR` keeps this run separate from the committed `model/` folder.
`TUNE = True` repeats the full grid search (about 2-3 minutes on a 16-core laptop, longer on a small machine);
`TUNE = False` trains with fixed hyper-parameters, is much faster and gives slightly different numbers.

In [ ]:
TUNE = True
OUT_DIR = PROJECT / "lab_run"
N_JOBS = max(1, min(joblib.cpu_count(), 8))

start = time.time()
metrics = tm.run(out_dir=OUT_DIR, tune=TUNE, n_jobs=N_JOBS)
print(f"\nFinished in {time.time() - start:.0f} s using {N_JOBS} worker(s).")

## 2. Comparison table (hold-out test split)

In [ ]:
metrics.set_index("ML Model Name").round(4)

## 3. Confusion matrices from the *saved* models

The models are reloaded from disk and score `test_data.csv` without any re-training.

In [ ]:
models = su.load_models(Path(OUT_DIR) / "model")
test = pd.read_csv(Path(OUT_DIR) / "test_data.csv")
X_test, y_test = su.prepare_features(test), su.prepare_target(test)

fig, axes = plt.subplots(1, 5, figsize=(22, 4))
for ax, (name, model) in zip(axes, models.items()):
    ConfusionMatrixDisplay.from_predictions(
        y_test, model.predict(X_test), display_labels=su.CLASS_NAMES, cmap="GnBu", colorbar=False, ax=ax
    )
    ax.set_title(name)
plt.tight_layout()
plt.show()

check = su.metrics_table(su.evaluate_all(models, X_test, y_test))
print("Reloaded models reproduce the table above:", np.allclose(check[su.METRICS], metrics[su.METRICS]))

## 4. Launch the app

```
streamlit run app.py
```

Upload `test_data.csv` (or any CSV with the same columns) in the sidebar to evaluate every saved model.